# 05 Reproduce from saved signals
## Цель ноутбука
Показать воспроизводимость: повторный прогон на одном и том же `signals.npz`.

## Почему это важно
Честное сравнение декодеров требует одинаковых входных сигналов.


In [ ]:
# [1/8] Импорты и поиск корня проекта
from pathlib import Path
import sys
import yaml
import pandas as pd
from IPython.display import display, Image

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    while cur != cur.parent:
        if (cur / 'pyproject.toml').exists():
            return cur
        cur = cur.parent
    raise RuntimeError('Repo root not found')

ROOT = find_repo_root(Path.cwd())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('[1/8] Импорты загружены')
print('ROOT =', ROOT)


# Настройка параметров эксперимента


In [ ]:
# [2/8] Параметры эксперимента (можно менять прямо здесь)
PARAMS = {
    'run_name': 'notebook_reproduce',
    'K': 64,
    'num_blocks': 12,
    'snr_db_list': [0, 2],
    'seed': 123,
    'decoders': ['viterbi', 'bcjr', 'neural_viterbi', 'neural_bcjr'],
    'epochs': 1,
    'learning_rate': 1e-3,
    'hidden_dim': 16,
    'reuse_saved_signals': False,
    'training_enabled': True,
}
print('[2/8] Параметры заданы')
display(pd.Series(PARAMS))


In [ ]:
# [3/8] Загружаем YAML, меняем параметры и сохраняем notebook-конфиг
from comm_ai.utils.io import load_yaml

cfg = load_yaml(ROOT / 'src/comm_ai/config/experiments/awgn_small.yaml')
cfg['experiment']['run_name'] = PARAMS['run_name']
cfg['experiment']['K'] = PARAMS['K']
cfg['experiment']['num_blocks'] = PARAMS['num_blocks']
cfg['experiment']['snr_db_list'] = PARAMS['snr_db_list']
cfg['experiment']['seed'] = PARAMS['seed']
cfg['experiment']['decoders'] = PARAMS['decoders']
cfg['experiment']['reuse_saved_signals'] = PARAMS['reuse_saved_signals']

cfg['training']['enabled'] = PARAMS['training_enabled']
cfg['training']['epochs'] = PARAMS['epochs']
cfg['training']['learning_rate'] = PARAMS['learning_rate']
cfg['training']['hidden_dim'] = PARAMS['hidden_dim']

nb_cfg_path = ROOT / 'outputs/runs' / f"{PARAMS['run_name']}_notebook_config.yaml"
nb_cfg_path.parent.mkdir(parents=True, exist_ok=True)
nb_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('[3/8] Конфиг сохранён в', nb_cfg_path)


In [ ]:
# [4/8] Первый запуск: генерируем сигнал
from comm_ai.experiments.run_experiment import run
cfg['experiment']['reuse_saved_signals'] = False
nb_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
out_first = run(str(nb_cfg_path))
print('[4/8] Первый запуск завершён')


In [ ]:
# [5/8] Второй запуск: переиспользуем сохранённый signals.npz
cfg['experiment']['reuse_saved_signals'] = True
nb_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
out_second = run(str(nb_cfg_path))
print('[5/8] Второй запуск завершён')


In [ ]:
# [6/8] Сравниваем результаты
r1 = pd.read_csv(out_first / 'results.csv')
r2 = pd.read_csv(out_second / 'results.csv')
print('Results identical:', r1.equals(r2))
display(r1)


### Что означает Results identical: True
Результаты совпали, значит pipeline воспроизводим на фиксированных сигналах.


In [ ]:
# [7/8] Показываем графики второго запуска
out_dir = out_second
print('BER/FER/Timing для воспроизводимого сценария')
display(Image(filename=str(out_dir / 'ber_plot.png')))
display(Image(filename=str(out_dir / 'fer_plot.png')))
display(Image(filename=str(out_dir / 'timing_plot.png')))


## Итог
Вы можете взять старый `signals.npz` и запускать новые декодеры на той же выборке для честного сравнения.
